# Bayesian Networks using Python (pgmpy)

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand Bayesian networks structure and semantics
- Build Bayesian networks in Python (a small pure-Python implementation that always runs, plus the equivalent pgmpy code for when that library is installed)
- Implement exact inference by enumeration
- Apply Bayesian networks to real-world decision-making problems

## 🔗 Where this fits

**Builds on:** Course 02 — Unit 3, lesson 01 "Learning under Uncertainty" — one Bayes update becomes a whole graph of them.

**Used later in:** Course 04 (AIAT 114) — Unit 3, where Naive Bayes is the simplest such network put to work as a classifier.

---

This notebook covers practical activities from **Course 02, Unit 3**:
- Building Bayesian networks in Python (pure-Python implementation, with the pgmpy library API shown alongside)
- Implementing inference algorithms for Bayesian networks (inference by enumeration)

---

## Introduction to Bayesian Networks

**Bayesian Networks** are probabilistic graphical models that represent conditional dependencies among variables using directed acyclic graphs (DAGs).


## 🌍 The case: two networks that shipped

**Xbox Live, from 2005.** **TrueSkill** — Herbrich, Minka & Graepel, *TrueSkill™: A Bayesian Skill Rating System* (NIPS 2006) — treats every player's skill as a hidden variable with a *distribution*, not a number: a mean and an uncertainty. Match results are the evidence; the posterior is updated by approximate message passing on a factor graph, which is the same conditional-probability machinery you are about to build. It was evaluated on match data from the Halo 2 beta and has run continuously on Xbox Live since 2005, handling matchmaking and leaderboards for millions of games. Every match you finish is one Bayesian update.

**An operating theatre, 1989.** The **ALARM** network — Beinlich, Suermondt, Chavez & Cooper, *A Logical Alarm Reduction Mechanism* — modelled a patient under anaesthesia with **37 variables and 46 arcs**: 8 diagnoses, 16 clinical measurements and 13 intermediate variables. Its purpose was to tell a real emergency from a monitoring artefact, so the theatre would stop drowning in false alarms. Thirty-five years later it is still the standard benchmark for Bayesian-network structure-learning algorithms.

The toy burglary alarm you will build in Part 3 shares more than a name with ALARM: same shape, same arithmetic, five variables instead of thirty-seven.

### What goes wrong without this — count the numbers

Take the five variables of Part 3: Burglary, Earthquake, Alarm, JohnCalls, MaryCalls. Written as a full joint probability table, that is 2⁵ = 32 combinations, so **31 free numbers**, every one of which somebody must estimate from data or elicit from an expert.

The network needs **ten**: 1 for Burglary, 1 for Earthquake, 4 for Alarm (one per combination of its two parents), 2 for JohnCalls, 2 for MaryCalls. The saving comes entirely from the arrows you *did not draw* — the claim that John's phone call depends on the alarm and on nothing else.

Now scale it. ALARM has 37 variables with two to four values each: the full joint table would need far more than 2³⁷ ≈ **137 billion** entries, which is not a hard estimation problem, it is an impossible one. The factorised network needs a few hundred. That is why Bayesian networks exist — not to be elegant, but to make the estimation problem finite.

---


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# pgmpy is a popular Bayesian-network library. It is OPTIONAL here:
# everything below runs on a small pure-Python implementation, and the
# pgmpy equivalents run additionally whenever the library is installed.

try:
    from pgmpy.models import BayesianNetwork
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination
    print("✅ pgmpy imported successfully — library examples will run too!")
    HAS_PGMPY = True
except ImportError:
    print("⚠️  pgmpy not installed (optional). Install with: pip install pgmpy")
    print("   No problem: every example below runs on the pure-Python")
    print("   implementation in the next cell.")
    HAS_PGMPY = False


⚠️  pgmpy not installed (optional). Install with: pip install pgmpy
   No problem: every example below runs on the pure-Python
   implementation in the next cell.


In [2]:
# A minimal discrete Bayesian network with inference by enumeration.
# ~50 lines of pure Python - enough to really SEE how BN inference works.

from itertools import product

class DiscreteBayesNet:
    """Discrete Bayesian network + exact inference by enumeration."""

    def __init__(self, name):
        self.name = name
        self.variables = {}   # var -> list of states
        self.parents = {}     # var -> list of parent variable names
        self.cpts = {}        # var -> {tuple(parent states): {state: prob}}
        print(f"✅ Created Bayesian network: {name}")

    def add_variable(self, name, states, parents=None, cpt=None):
        """Add a variable with its CPT: cpt[(parent values)][state] = prob.
        Root variables use the empty tuple () as key."""
        parents = parents or []
        self.variables[name] = list(states)
        self.parents[name] = parents
        self.cpts[name] = cpt
        suffix = f", parents={parents}" if parents else " (root)"
        print(f"  ➕ Added variable: {name} states={states}{suffix}")

    def prob(self, var, value, assignment):
        """P(var = value | its parents' values in this assignment)"""
        key = tuple(assignment[p] for p in self.parents[var])
        return self.cpts[var][key][value]

    def joint(self, assignment):
        """Chain rule: P(full assignment) = product of P(var | parents)"""
        p = 1.0
        for var in self.variables:
            p *= self.prob(var, assignment[var], assignment)
        return p

    def query(self, var, evidence=None):
        """P(var | evidence), by summing the joint over all hidden variables.
        This is 'inference by enumeration' - simple and exact (but exponential;
        libraries like pgmpy use faster algorithms such as Variable Elimination)."""
        evidence = evidence or {}
        hidden = [v for v in self.variables if v != var and v not in evidence]
        totals = {}
        for target_state in self.variables[var]:
            total = 0.0
            for combo in product(*(self.variables[h] for h in hidden)):
                assignment = dict(evidence)
                assignment[var] = target_state
                assignment.update(zip(hidden, combo))
                total += self.joint(assignment)
            totals[target_state] = total
        norm = sum(totals.values())
        return {s: t / norm for s, t in totals.items()}

def print_dist(title, dist):
    print(f"   {title}")
    for state, p in dist.items():
        print(f"      {state:<5} {p:.4f}")

print("✅ DiscreteBayesNet ready (pure Python, no libraries needed)")

✅ DiscreteBayesNet ready (pure Python, no libraries needed)


## Part 1: Building a Simple Bayesian Network

Let's create a simple medical diagnosis Bayesian network.


In [3]:
# Example: Medical Diagnosis Network
# Structure: Disease -> Symptom
# P(Disease=Yes) = 0.1 (10% of patients have the disease)
# P(Symptom=Yes | Disease=Yes) = 0.9, P(Symptom=Yes | Disease=No) = 0.2

print("=" * 60)
print("Bayesian Network: Medical Diagnosis")
print("=" * 60)

medical = DiscreteBayesNet("Medical Diagnosis")

# Prior for Disease (root variable -> CPT key is the empty tuple)
medical.add_variable('Disease', ['No', 'Yes'],
                     cpt={(): {'No': 0.9, 'Yes': 0.1}})

# Symptom depends on Disease
medical.add_variable('Symptom', ['No', 'Yes'], parents=['Disease'],
                     cpt={('No',):  {'No': 0.8, 'Yes': 0.2},   # healthy: 20% show symptom
                          ('Yes',): {'No': 0.1, 'Yes': 0.9}})  # diseased: 90% show symptom

print("\nStructure: Disease -> Symptom")
print("Chain rule check: P(Disease=Yes, Symptom=Yes) =", end=" ")
print(f"{medical.joint({'Disease': 'Yes', 'Symptom': 'Yes'}):.4f}  (= 0.1 x 0.9)")

# The same model in pgmpy (runs only if the library is installed)
if HAS_PGMPY:
    model = BayesianNetwork([('Disease', 'Symptom')])
    cpd_disease = TabularCPD(variable='Disease', variable_card=2,
                             values=[[0.9],   # P(Disease = No)  = 0.9
                                     [0.1]],  # P(Disease = Yes) = 0.1
                             state_names={'Disease': ['No', 'Yes']})
    cpd_symptom = TabularCPD(variable='Symptom', variable_card=2,
                             evidence=['Disease'], evidence_card=[2],
                             values=[[0.8, 0.1],   # P(Symptom=No | Disease=No/Yes)
                                     [0.2, 0.9]],  # P(Symptom=Yes | Disease=No/Yes)
                             state_names={'Symptom': ['No', 'Yes'],
                                          'Disease': ['No', 'Yes']})
    model.add_cpds(cpd_disease, cpd_symptom)
    print(f"\npgmpy version of the same model — valid: {model.check_model()}")
else:
    print("\n(pgmpy not installed — the pure-Python model above is the one we use.)")

Bayesian Network: Medical Diagnosis
✅ Created Bayesian network: Medical Diagnosis
  ➕ Added variable: Disease states=['No', 'Yes'] (root)
  ➕ Added variable: Symptom states=['No', 'Yes'], parents=['Disease']

Structure: Disease -> Symptom
Chain rule check: P(Disease=Yes, Symptom=Yes) = 0.0900  (= 0.1 x 0.9)

(pgmpy not installed — the pure-Python model above is the one we use.)


## Part 2: Bayesian Inference

Now let's perform inference: given evidence (symptom), what's the probability of disease?


In [4]:
# Inference: given evidence (symptom), what is the probability of disease?

print("=" * 60)
print("Bayesian Inference Examples:")
print("=" * 60)

# Query 1: Prior probability of disease (no evidence)
print("\n1. Prior Probability of Disease:")
prior = medical.query('Disease')
print_dist("P(Disease):", prior)

# Query 2: Posterior probability given the symptom is observed
print("\n2. Posterior Probability: P(Disease | Symptom = Yes)")
posterior = medical.query('Disease', evidence={'Symptom': 'Yes'})
print_dist("P(Disease | Symptom=Yes):", posterior)

# Query 3: Marginal probability of the symptom
print("\n3. Marginal Probability: P(Symptom)")
marginal = medical.query('Symptom')
print_dist("P(Symptom):", marginal)

# Hand-check with Bayes' theorem (all computed, nothing hardcoded):
p_s_yes = 0.9 * 0.2 + 0.1 * 0.9                # P(S=Yes) = sum over Disease
p_d_given_s = (0.9 * 0.1) / p_s_yes            # P(D=Yes|S=Yes) = P(S|D)P(D)/P(S)
print("\n✔ Bayes'-theorem hand check:")
print(f"   P(Symptom=Yes) = 0.9x0.2 + 0.1x0.9 = {p_s_yes:.4f}")
print(f"   P(Disease=Yes | Symptom=Yes) = 0.1x0.9 / {p_s_yes:.4f} = {p_d_given_s:.4f}")
print(f"   Matches the enumeration result: {abs(p_d_given_s - posterior['Yes']) < 1e-12}")

# pgmpy Variable Elimination gives the same answers (if installed)
if HAS_PGMPY:
    inference = VariableElimination(model)
    print("\npgmpy (Variable Elimination) cross-check:")
    print(inference.query(variables=['Disease'], evidence={'Symptom': 'Yes'}))
else:
    print("\n(pgmpy not installed — enumeration results above are exact and complete.)")

Bayesian Inference Examples:

1. Prior Probability of Disease:
   P(Disease):
      No    0.9000
      Yes   0.1000

2. Posterior Probability: P(Disease | Symptom = Yes)
   P(Disease | Symptom=Yes):
      No    0.6667
      Yes   0.3333

3. Marginal Probability: P(Symptom)
   P(Symptom):
      No    0.7300
      Yes   0.2700

✔ Bayes'-theorem hand check:
   P(Symptom=Yes) = 0.9x0.2 + 0.1x0.9 = 0.2700
   P(Disease=Yes | Symptom=Yes) = 0.1x0.9 / 0.2700 = 0.3333
   Matches the enumeration result: True

(pgmpy not installed — enumeration results above are exact and complete.)


## Part 3: More Complex Bayesian Network

Let's build a more complex network with multiple variables.


In [5]:
# Example: the classic Alarm network (Russell & Norvig)
# Structure: Burglary -> Alarm <- Earthquake ; Alarm -> JohnCalls, MaryCalls

print("=" * 60)
print("Complex Bayesian Network: Alarm System")
print("=" * 60)

alarm_net = DiscreteBayesNet("Alarm System")

alarm_net.add_variable('Burglary',   ['No', 'Yes'], cpt={(): {'No': 0.999, 'Yes': 0.001}})
alarm_net.add_variable('Earthquake', ['No', 'Yes'], cpt={(): {'No': 0.998, 'Yes': 0.002}})

# Alarm depends on both Burglary and Earthquake: cpt[(burglary, earthquake)]
alarm_net.add_variable('Alarm', ['No', 'Yes'], parents=['Burglary', 'Earthquake'],
    cpt={('No',  'No'):  {'No': 0.999, 'Yes': 0.001},
         ('No',  'Yes'): {'No': 0.71,  'Yes': 0.29},
         ('Yes', 'No'):  {'No': 0.06,  'Yes': 0.94},
         ('Yes', 'Yes'): {'No': 0.05,  'Yes': 0.95}})

alarm_net.add_variable('JohnCalls', ['No', 'Yes'], parents=['Alarm'],
    cpt={('No',):  {'No': 0.95, 'Yes': 0.05},
         ('Yes',): {'No': 0.10, 'Yes': 0.90}})

alarm_net.add_variable('MaryCalls', ['No', 'Yes'], parents=['Alarm'],
    cpt={('No',):  {'No': 0.99, 'Yes': 0.01},
         ('Yes',): {'No': 0.30, 'Yes': 0.70}})

# Inference: both neighbors call - was it a burglary?
print("\nInference: P(Burglary | JohnCalls=Yes, MaryCalls=Yes)")
result = alarm_net.query('Burglary', evidence={'JohnCalls': 'Yes', 'MaryCalls': 'Yes'})
print_dist("P(Burglary | both call):", result)

print("\nInference: P(Alarm | JohnCalls=Yes, MaryCalls=Yes)")
result_alarm = alarm_net.query('Alarm', evidence={'JohnCalls': 'Yes', 'MaryCalls': 'Yes'})
print_dist("P(Alarm | both call):", result_alarm)

print(f"\n💡 Interpretation (computed above): even with BOTH neighbors calling,")
print(f"   P(Burglary=Yes) is only {result['Yes']:.3f} — because burglaries are rare")
print(f"   (prior 0.001) and the alarm can also be triggered by earthquakes.")

Complex Bayesian Network: Alarm System
✅ Created Bayesian network: Alarm System
  ➕ Added variable: Burglary states=['No', 'Yes'] (root)
  ➕ Added variable: Earthquake states=['No', 'Yes'] (root)
  ➕ Added variable: Alarm states=['No', 'Yes'], parents=['Burglary', 'Earthquake']
  ➕ Added variable: JohnCalls states=['No', 'Yes'], parents=['Alarm']
  ➕ Added variable: MaryCalls states=['No', 'Yes'], parents=['Alarm']

Inference: P(Burglary | JohnCalls=Yes, MaryCalls=Yes)
   P(Burglary | both call):
      No    0.7158
      Yes   0.2842

Inference: P(Alarm | JohnCalls=Yes, MaryCalls=Yes)
   P(Alarm | both call):
      No    0.2393
      Yes   0.7607

💡 Interpretation (computed above): even with BOTH neighbors calling,
   P(Burglary=Yes) is only 0.284 — because burglaries are rare
   (prior 0.001) and the alarm can also be triggered by earthquakes.


## 💬 Discuss

1. **Where did `P(Disease=Yes) = 0.10` come from?** It was typed into the cell. Suppose this network is deployed in a specialist referral clinic where 40% of arriving patients really do have the disease. Re-run the query with that prior and you get **P(Disease | Symptom=Yes) = 0.75** instead of 0.33 — the same symptom, the same test, more than double the posterior. Should a diagnostic tool ship with a fixed prior baked in, ship with a prior the site configures, or refuse to give a number until it has local data? Who is accountable for the choice?
2. **The alarm network can "explain away".** Learn that an earthquake happened, and the probability of burglary given the alarm goes *down* — even though no arrow connects Earthquake to Burglary. Nobody programmed that; it falls out of the structure. Is a model that produces conclusions its author never wrote down a strength or a hazard? What would you check before letting it drive a decision?
3. **pgmpy was not installed when this ran, so the 50-line pure-Python implementation did the work — and got the same answers**, hand-checked against Bayes' theorem in the output. When is writing your own implementation the right professional call, and when is it a liability you are handing to whoever maintains this after you? Give a concrete criterion, not "it depends".

---


## ⚠️ Where this breaks

- **Inference by enumeration is exponential, and that is what you just ran.** `DiscreteBayesNet.query` sums the joint over every combination of the hidden variables. Five binary variables: 32 terms, instant. ALARM's 37 variables: not in this universe. Real systems use variable elimination or junction trees — which is what pgmpy would have done — but there is no general escape: exact inference in Bayesian networks is NP-hard (Cooper, 1990), and so is guaranteed-accurate approximate inference (Dagum & Luby, 1993). What makes real networks tractable is their *shape*, not the algorithm.
- **Every number in this notebook was typed by a human.** A node with two binary parents needs 4 rows; with four parents, 16; with six, 64 — each estimated from data you may not have, or elicited from an expert who may be guessing. Parameter estimation, not inference, is where most Bayesian-network projects actually die.
- **The arrows assert conditional independencies, and if they are wrong the answer is confidently wrong.** Going from 31 numbers to 10 is bought by claiming that JohnCalls is independent of Burglary *given* Alarm. If John also hears the neighbour's dog whenever there is a burglar, that claim is false, and the network will still return a crisp posterior with no complaint.
- **An arrow is not a cause.** The graph is a factorisation of a joint distribution. Disease → Symptom and Symptom → Disease can represent the *same* joint distribution with different tables. Reading arrows as causal claims — and then reasoning about interventions with them — is a real and consequential error; it is exactly the distinction Pearl (1988, reference 1) exists to make precise.
- **Learning the structure from data is harder than using it.** Finding the highest-scoring DAG for a dataset is NP-hard (Chickering, 1996), which is why reference 3 is an entire survey rather than a recipe.
- **Discrete tables only.** Everything here is a finite conditional probability table. Continuous quantities — blood pressure, latency, price — need conditional Gaussians, discretisation (which throws information away) or sampling.
- **A calibrated belief is not a licence to act.** Microsoft Research's **Lumière** project (Horvitz *et al.*, UAI 1998) built Bayesian user models to infer what a user was trying to do; Horvitz has described that work as the basis for the Office Assistant that shipped in Microsoft Office. The Assistant became one of the most disliked features in software history — not because the posterior over "this user is writing a letter" was necessarily wrong, but because a well-calibrated belief says nothing about whether interrupting the user is a good idea. Inference and action are separate decisions.
- **The cheaper alternative.** One hypothesis and one piece of evidence needs no graph at all — that is the single Bayes update from notebook 01. Many features but only a class label to predict? Naive Bayes (Course 04, Unit 3) is this same network with every arrow pointing out from the class, and on real text and spam it is often good enough. Build the graph when the *dependencies between the evidence* are the thing you care about.

---


## Summary

### Key Concepts:
1. **Bayesian Networks**: Graphical models representing conditional dependencies
2. **CPDs / CPTs (Conditional Probability Distributions/Tables)**: Define relationships between variables
3. **Inference**: Compute posterior probabilities given evidence
4. **Inference by Enumeration**: The simple exact algorithm we implemented here — sum the joint distribution over hidden variables (libraries like pgmpy speed this up with Variable Elimination)

### Applications:
- Medical diagnosis
- Risk assessment
- Decision support systems
- Natural language processing

**Reference:** Course 02, Unit 3: "Building Bayesian networks using Python libraries (pgmpy)" and "Implementing inference algorithms for Bayesian networks" — implemented here in pure Python so it runs everywhere, with the equivalent pgmpy code included for environments that have the library installed.

## 📚 References

1. Pearl, J. (1988). *Probabilistic Reasoning in Intelligent Systems: Networks of Plausible Inference*. Morgan Kaufmann. (the book that established Bayesian networks)
2. Koller, D., & Friedman, N. (2009). *Probabilistic Graphical Models: Principles and Techniques*. MIT Press.
3. Kitson, N. K., Constantinou, A. C., Guo, Z., Liu, Y., & Chobtham, K. (2021). *A Survey of Bayesian Network Structure Learning*. arXiv preprint. <https://arxiv.org/abs/2109.11415>
4. Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.), Chapter 13 (Probabilistic Reasoning; the Alarm network used above comes from this book). Pearson. <https://aima.cs.berkeley.edu/>